# Scraped-Style Workout Recommendation ML Project
## Complete Preprocessing + Machine Learning Pipeline

This notebook is specifically designed for the uploaded scraped-style noisy dataset.


In [ ]:
# INSTALL REQUIRED LIBRARIES

!pip install pandas numpy scikit-learn xgboost shap matplotlib seaborn joblib

In [ ]:
# IMPORTS

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# LOAD DATASET

DATA_PATH = "scraped_style_noisy_workout_dataset.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset Shape:", df.shape)

df.head()

In [ ]:
# DATASET OVERVIEW

print(df.info())

print(df.isnull().sum())

print(df.describe(include='all'))

# Preprocessing Problems Intentionally Added

The dataset contains realistic scraped-data issues:
- Missing values
- Duplicate rows
- Outliers
- Invalid categories
- Corrupted text
- Scraping artifacts


In [ ]:
# REMOVE DUPLICATES

print("Before removing duplicates:", len(df))

df.drop_duplicates(inplace=True)

print("After removing duplicates:", len(df))

In [ ]:
# HANDLE MISSING VALUES

numeric_cols = df.select_dtypes(include=['int64','float64']).columns
categorical_cols = df.select_dtypes(include=['object']).columns

# Numerical imputation
num_imputer = SimpleImputer(strategy='median')

df[numeric_cols] = num_imputer.fit_transform(df[numeric_cols])

# Categorical imputation
cat_imputer = SimpleImputer(strategy='most_frequent')

df[categorical_cols] = cat_imputer.fit_transform(df[categorical_cols])

print(df.isnull().sum().sum())

In [ ]:
# CLEAN INVALID TEXT VALUES

invalid_values = [
    '[deleted]',
    '[removed]',
    'NULL',
    'scrape_error',
    '404_error',
    '{}',
    '[]',
    'NoneType',
    '???',
    'missing'
]

for col in df.select_dtypes(include='object').columns:

    df[col] = df[col].astype(str)

    for val in invalid_values:
        df[col] = df[col].str.replace(val, 'unknown', regex=False)

df.head()

In [ ]:
# HANDLE OUTLIERS

# Example columns
outlier_cols = [
    'user_age',
    'user_bmi',
    'sleep_hours',
    'target_duration_min'
]

for col in outlier_cols:

    if col in df.columns:

        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)

        iqr = q3 - q1

        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        df = df[
            (df[col] >= lower) &
            (df[col] <= upper)
        ]

print(df.shape)

In [ ]:
# CLEAN WHITESPACES

for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip()

df.head()

In [ ]:
# TARGET COLUMN

target_col = 'target_workout_type'

print(df[target_col].value_counts())

In [ ]:
# ENCODE TARGET

target_encoder = LabelEncoder()

df['target_encoded'] = target_encoder.fit_transform(
    df['target_workout_type']
)

print(target_encoder.classes_)

In [ ]:
# FEATURE SELECTION

drop_cols = [
    'target_workout_type',
    'target_encoded'
]

FEATURES = [
    col for col in df.columns
    if col not in drop_cols
]

X = df[FEATURES]
y = df['target_encoded']

print(X.shape)
print(y.shape)

In [ ]:
# ENCODE CATEGORICAL FEATURES

categorical_cols = X.select_dtypes(include='object').columns

encoders = {}

for col in categorical_cols:

    le = LabelEncoder()

    X[col] = le.fit_transform(
        X[col].astype(str)
    )

    encoders[col] = le

X.head()

In [ ]:
# TRAIN TEST SPLIT

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

In [ ]:
# FEATURE SCALING

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# MACHINE LEARNING MODELS

models = {

    'Logistic Regression':
        LogisticRegression(max_iter=1000),

    'Decision Tree':
        DecisionTreeClassifier(max_depth=8),

    'Random Forest':
        RandomForestClassifier(
            n_estimators=300,
            random_state=42
        ),

    'XGBoost':
        XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            eval_metric='mlogloss',
            random_state=42
        )
}

In [ ]:
# TRAIN AND EVALUATE

results = []

for name, model in models.items():

    print('=' * 60)
    print(name)
    print('=' * 60)

    if name == 'Logistic Regression':

        model.fit(X_train_scaled, y_train)

        preds = model.predict(X_test_scaled)

    else:

        model.fit(X_train, y_train)

        preds = model.predict(X_test)

    accuracy = accuracy_score(y_test, preds)
    precision = precision_score(y_test, preds, average='weighted')
    recall = recall_score(y_test, preds, average='weighted')
    f1 = f1_score(y_test, preds, average='weighted')

    print(classification_report(y_test, preds))

    results.append({
        'Model': name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1': f1
    })

In [ ]:
# RESULTS TABLE

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by='F1',
    ascending=False
)

results_df

In [ ]:
# BEST MODEL

best_model_name = results_df.iloc[0]['Model']

print("Best Model:", best_model_name)

In [ ]:
# TRAIN FINAL XGBOOST MODEL

final_model = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    eval_metric='mlogloss',
    random_state=42
)

final_model.fit(X_train, y_train)

final_preds = final_model.predict(X_test)

print(classification_report(y_test, final_preds))

In [ ]:
# CONFUSION MATRIX

cm = confusion_matrix(y_test, final_preds)

plt.figure(figsize=(8,6))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.title('Confusion Matrix')

plt.xlabel('Predicted')
plt.ylabel('Actual')

plt.show()

In [ ]:
# FEATURE IMPORTANCE

importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': final_model.feature_importances_
})

importance_df = importance_df.sort_values(
    by='Importance',
    ascending=False
)

importance_df.head(20)

In [ ]:
# FEATURE IMPORTANCE PLOT

plt.figure(figsize=(10,8))

sns.barplot(
    data=importance_df.head(15),
    x='Importance',
    y='Feature'
)

plt.title('Top Feature Importance')

plt.show()

# Research Conclusion

This notebook demonstrates:
- realistic scraped-data preprocessing
- missing value handling
- duplicate removal
- outlier detection
- categorical cleaning
- machine learning training
- model evaluation
- explainable feature importance
